In [ ]:
# Cell 1: List input files — confirms dataset paths
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Cell 2: Pull latest training scripts from GitHub
# Syncs all Janus AI files so Kaggle always runs the latest version.
import urllib.request, os, sys, importlib

GITHUB_RAW = 'https://raw.githubusercontent.com/Thoseidiots/Janus/main'

# Flat files — downloaded to /kaggle/working/
FILES_TO_SYNC = [
    'train_avus_kaggle.py',
    'avus.py',
    'avus_eval.py',
    'avus_blt.py',
    'collaborative_trainer.py',
    'avus_tokenizer_trainer.py',
    'screen_action_dataset_generator.py',
    'skill_curriculum.py',
    'growing_avus.py',
    'janus_optimizer.py',
    'holographic_gradient_battery.py',
    'gradient_battery.py',
    'deep_training_data.py',
    'procedural_dataset.py',
    # Phase 2 rendering quality modules
    'temporal_consistency_dataset.py',
    'spatial_detail_dataset.py',
    'geometric_constraint_dataset.py',
    'optical_flow_dataset.py',
    'semantic_lighting_dataset.py',
    # Audio validator
    'audio_validator.py',
]

# Subdirectory files — downloaded preserving folder structure
SUBDIR_FILES_TO_SYNC = [
    # Human-like language dataset generator
    'human_dataset/generate_human_dataset.py',
    # Synthetic e-commerce dataset generators
    'synthetic_database/generate_dataset.py',
    'synthetic_database/generate_rich_training.py',
    'synthetic_database/schema.sql',
    'synthetic_database/relationships.json',
    # Pre-generated training pairs (avoids runtime generation cost)
    'human_dataset/output/avus_human_pairs.txt',
    'synthetic_database/output/avus_training_pairs.txt',
]

os.makedirs('/kaggle/working', exist_ok=True)

# Sync flat files
for fname in FILES_TO_SYNC:
    url  = f'{GITHUB_RAW}/{fname}'
    dest = f'/kaggle/working/{fname}'
    try:
        urllib.request.urlretrieve(url, dest)
        size = os.path.getsize(dest)
        print(f'[sync] {fname} ({size:,} bytes) <- GitHub')
    except Exception as e:
        print(f'[sync] WARNING: could not fetch {fname}: {e}')

# Sync subdirectory files — create dirs as needed
for fpath in SUBDIR_FILES_TO_SYNC:
    url  = f'{GITHUB_RAW}/{fpath}'
    dest = f'/kaggle/working/{fpath}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dest)
        size = os.path.getsize(dest)
        print(f'[sync] {fpath} ({size:,} bytes) <- GitHub')
    except Exception as e:
        print(f'[sync] WARNING: could not fetch {fpath}: {e}')

# /kaggle/working MUST be first so synced files override dataset snapshot
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
else:
    # Move it to front if already present
    sys.path.remove('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')

# Force-reload avus and avus_blt so we get the GitHub version, not the cached dataset version
for mod_name in ['avus', 'avus_blt', 'collaborative_trainer', 'train_avus_kaggle']:
    if mod_name in sys.modules:
        try:
            importlib.reload(sys.modules[mod_name])
            print(f'[sync] Reloaded {mod_name}')
        except Exception as e:
            print(f'[sync] Could not reload {mod_name}: {e}')

print('[sync] Done. /kaggle/working is first in sys.path.')


In [ ]:
# Cell 3: Clean up leftover staging dirs
import os, shutil
staging_path = '/kaggle/working/upload_staging'
if os.path.exists(staging_path):
    shutil.rmtree(staging_path)
    print('Cleaned up staging area.')
else:
    print('No staging area to clean.')

In [ ]:
# Cell 4: Patch AvusConfig for backward compatibility + device-aware nn.Linear patch
# Adds missing fields to old AvusConfig from dataset snapshot
import sys, os

# Ensure /kaggle/working (GitHub-synced) takes priority
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')
else:
    sys.path.remove('/kaggle/working')
    sys.path.insert(0, '/kaggle/working')

# Patch AvusConfig to add missing fields if using old dataset version
try:
    from avus import AvusConfig as _AC
    if not hasattr(_AC, 'window_size') or 'window_size' not in _AC.__dataclass_fields__:
        # Old version — monkey-patch missing fields with defaults
        import dataclasses
        _AC.window_size       = dataclasses.field(default=0)
        _AC.n_experts         = dataclasses.field(default=0)
        _AC.n_experts_active  = dataclasses.field(default=2)
        _AC.draft_config      = dataclasses.field(default=None)
        # Also patch instances via __getattr__
        _orig_getattr = _AC.__getattribute__
        def _safe_getattr(self, name):
            _defaults = {'window_size': 0, 'n_experts': 0, 'n_experts_active': 2, 'draft_config': None}
            try:
                return _orig_getattr(self, name)
            except AttributeError:
                if name in _defaults:
                    return _defaults[name]
                raise
        _AC.__getattribute__ = _safe_getattr
        print('[compat] AvusConfig patched with window_size, n_experts defaults')
    else:
        print('[compat] AvusConfig already has window_size — no patch needed')
except Exception as e:
    print(f'[compat] AvusConfig patch failed: {e}')

# Cell 4: Apply device-aware nn.Linear patch ONCE (prevents recursion)
import torch.nn as nn

if not hasattr(nn.Linear, '_is_patched'):
    _REAL_LINEAR_FWD = nn.Linear.forward

    def _device_aware_linear(self, x):
        if self.weight.device != x.device:
            self.weight = nn.Parameter(self.weight.to(x.device),
                                       requires_grad=self.weight.requires_grad)
        if self.bias is not None and self.bias.device != x.device:
            self.bias = nn.Parameter(self.bias.to(x.device),
                                     requires_grad=self.bias.requires_grad)
        return _REAL_LINEAR_FWD(self, x)

    nn.Linear.forward = _device_aware_linear
    nn.Linear._is_patched = True
    print('[patch] Device-aware nn.Linear applied.')
else:
    print('[patch] nn.Linear already patched — skipping.')

In [ ]:
# Cell 5: Locate the Janus repo (dataset) and add to sys.path
# /kaggle/working takes priority (has GitHub-synced files)
import os, sys

REPO = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'avus.py' in files:
        REPO = root
        break

if REPO is None:
    raise FileNotFoundError('Could not find avus.py in /kaggle/input')

print(f'Repo (dataset) found at: {REPO}')
if REPO not in sys.path:
    sys.path.append(REPO)  # append — working dir takes priority

# Confirm which train_avus_kaggle.py will be used
import importlib.util
for p in sys.path:
    candidate = os.path.join(p, 'train_avus_kaggle.py')
    if os.path.exists(candidate):
        print(f'[train] Will use: {candidate}')
        break

In [ ]:
# Cell 6: Patch avus.py RMSNorm to be device-aware
import os, sys

# Find avus.py — prefer dataset version (it's the source)
REPO = [p for p in sys.path if os.path.exists(os.path.join(p, 'avus.py'))][-1]  # last = dataset
avus_src = open(os.path.join(REPO, 'avus.py')).read()

OLD_NORM = '        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n        return x * norm * self.weight'
NEW_NORM = ('        if self.weight.device != x.device:\n'
            '            self.weight = torch.nn.Parameter(self.weight.to(x.device), requires_grad=self.weight.requires_grad)\n'
            '        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n'
            '        return x * norm * self.weight')

if OLD_NORM in avus_src:
    avus_src = avus_src.replace(OLD_NORM, NEW_NORM)
    print('[patch] RMSNorm device-aware patch applied.')
else:
    print('[patch] RMSNorm already patched or pattern not found.')

with open('/kaggle/working/avus.py', 'w') as f:
    f.write(avus_src)
print('[patch] Patched avus.py written to /kaggle/working/avus.py')

In [ ]:
# Cell 7: Set up Kaggle API credentials from secret
import os, json
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('Kiro')
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    creds = {'username': 'ishmaelsears', 'key': token}
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump(creds, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('[kaggle] API credentials configured from secret.')
except Exception as e:
    print(f'[kaggle] No Kiro secret ({e}) — auto-push to dataset disabled.')

In [ ]:
# Cell: Pre-tokenize training data once and cache to tokens.pt
# On first run this takes ~2-3 minutes. Every subsequent run loads instantly.
import os, sys, torch, importlib.util
from pathlib import Path

TOKENS_OUT = Path('/kaggle/working/tokens.pt')

if TOKENS_OUT.exists():
    size_mb = TOKENS_OUT.stat().st_size / 1e6
    print(f'[pretok] tokens.pt already exists ({size_mb:.1f} MB) — skipping generation')
else:
    print('[pretok] Generating and tokenizing training data...')
    from train_avus_kaggle import (
        generate_screen_action_pairs,
        generate_3d_pairs,
        generate_language_pairs,
        generate_reasoning_pairs,
        AvusTokenizer,
        SAMPLES_PER_DATASET,
    )

    tokenizer = AvusTokenizer()
    eos_id = tokenizer.encode('<|endoftext|>')[0]

    all_texts = []
    all_texts += generate_screen_action_pairs(SAMPLES_PER_DATASET)
    all_texts += generate_3d_pairs(SAMPLES_PER_DATASET)
    all_texts += generate_language_pairs(SAMPLES_PER_DATASET)
    all_texts += generate_reasoning_pairs(SAMPLES_PER_DATASET)

    # Human-like language dataset (28k pairs)
    _human_txt = Path('/kaggle/working/human_dataset/output/avus_human_pairs.txt')
    if _human_txt.exists():
        _human_lines = [l.strip() for l in _human_txt.read_text(encoding='utf-8').splitlines() if l.strip()]
        all_texts += _human_lines
        print(f'[pretok] Human language data: {len(_human_lines):,} pairs')
    else:
        # Generate at runtime using the synced generator
        _hd_file = Path('/kaggle/working/human_dataset/generate_human_dataset.py')
        if _hd_file.exists():
            _spec = importlib.util.spec_from_file_location('_hd_gen', str(_hd_file))
            _hd = importlib.util.module_from_spec(_spec)
            _spec.loader.exec_module(_hd)
            _human_rt = (
                _hd.gen_opinions(2000) + _hd.gen_emotions(2000) + _hd.gen_small_talk(2500) +
                _hd.gen_stories(1666) + _hd.gen_curiosity(2000) + _hd.gen_humor(2000) +
                _hd.gen_uncertainty(3000) + _hd.gen_empathy(2000) + _hd.gen_debate(1666) +
                _hd.gen_aspirations(1666) + _hd.gen_reflection(2000) + _hd.gen_identity(2500)
            )
            all_texts += _human_rt
            print(f'[pretok] Human language data: {len(_human_rt):,} pairs (generated)')
        else:
            print('[pretok] Human language data: not found, skipping')

    # Synthetic e-commerce dataset (13k pairs)
    _synth_txt = Path('/kaggle/working/synthetic_database/output/avus_training_pairs.txt')
    if _synth_txt.exists():
        _synth_lines = [l.strip() for l in _synth_txt.read_text(encoding='utf-8').splitlines() if l.strip()]
        all_texts += _synth_lines
        print(f'[pretok] Synthetic e-commerce data: {len(_synth_lines):,} pairs')
    else:
        print('[pretok] Synthetic e-commerce data: not found, skipping')

    print(f'[pretok] Tokenizing {len(all_texts):,} sequences...')
    all_tokens = []
    for text in all_texts:
        toks = tokenizer.encode(text)
        if toks:
            all_tokens.extend(toks)
            if all_tokens[-1] != eos_id:
                all_tokens.append(eos_id)

    token_tensor = torch.tensor(all_tokens, dtype=torch.long)
    torch.save({
        'tokens': token_tensor,
        'sources': {
            'screen_actions': SAMPLES_PER_DATASET,
            '3d': SAMPLES_PER_DATASET,
            'language': SAMPLES_PER_DATASET,
            'reasoning': SAMPLES_PER_DATASET,
            'human_language': len([t for t in all_texts if 'Human:' in t or 'Avus:' in t]),
            'ecommerce': len([t for t in all_texts if 'order' in t.lower() or 'shipment' in t.lower()]),
        }
    }, str(TOKENS_OUT))

    size_mb = TOKENS_OUT.stat().st_size / 1e6
    print(f'[pretok] Saved {len(all_tokens):,} tokens to {TOKENS_OUT} ({size_mb:.1f} MB)')
    print('[pretok] Training will load this file instantly on next run.')


In [ ]:
# Cell 8: Run training
# Uses the GitHub-synced train_avus_kaggle.py from /kaggle/working
# Applies two runtime patches:
#   1. Forces KAGGLE_MODE = True
#   2. Fixes focal loss targets device mismatch
import os, sys

# Find the training script — /kaggle/working first (GitHub-synced)
TRAIN_SCRIPT = None
for p in sys.path:
    candidate = os.path.join(p, 'train_avus_kaggle.py')
    if os.path.exists(candidate):
        TRAIN_SCRIPT = candidate
        break

if TRAIN_SCRIPT is None:
    raise FileNotFoundError('train_avus_kaggle.py not found in sys.path')

print(f'[train] Loading: {TRAIN_SCRIPT}')
script = open(TRAIN_SCRIPT).read()

# Patch 1: force Kaggle mode on
script = script.replace(
    'KAGGLE_MODE           = False',
    'KAGGLE_MODE           = True'
)

# Patch 2: fix focal loss targets device
script = script.replace(
    '        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction="none")',
    '        targets = targets.to(logits.device)\n        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction="none")'
)

print('[train] Patches applied. Starting training...')
exec(script)

In [ ]:
# Cell 9: Show output files and download links
import os
from IPython.display import FileLink, display

working = '/kaggle/working'
print('Output files:')
for f in sorted(os.listdir(working)):
    path = os.path.join(working, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {f}  ({size_mb:.1f} MB)')

for fname in ['avus_1b_weights.pt', 'skill_chart.png', 'skill_state.json', 'hbm_weights.pt']:
    fpath = os.path.join(working, fname)
    if os.path.exists(fpath):
        display(FileLink(fname))
    else:
        print(f'  (not found: {fname})')